# Customer Segmentation & Prediction

End-to-end workflow: data validation, K-Means/Agglomerative/DBSCAN comparison, segment profiling, separate segment Random Forest models, GridSearchCV and business recommendations.

In [ ]:
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.impute import SimpleImputer
import json

## 1. Load and validate data

In [ ]:
df = pd.read_csv('segmentation_data.csv')
display(df.head())
print(df.shape); print(df.isna().sum())

## 2. Prepare features and compare clustering algorithms

Churn and CustomerID are excluded from clustering. Excluding Churn prevents target leakage.

In [ ]:
df['PaperlessBillingBinary'] = df['PaperlessBilling'].map({'Yes':1,'No':0})
num_features = ['Tenure','MonthlyCharges','TotalCharges','SeniorCitizen','PaperlessBillingBinary']
cat_features = ['Contract','PaymentMethod']
prep = ColumnTransformer([('num', Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]), num_features), ('cat',OneHotEncoder(handle_unknown='ignore'),cat_features)])
X = prep.fit_transform(df[num_features+cat_features])
if hasattr(X,'toarray'): X=X.toarray()
rows=[]
for k in range(2,7):
    for algorithm, labels, params in [('K-Means',KMeans(n_clusters=k,random_state=42,n_init=20).fit_predict(X),f'k={k}'),('Agglomerative',AgglomerativeClustering(n_clusters=k,linkage='ward').fit_predict(X),f'k={k}, linkage=ward')]:
        rows.append({'Algorithm':algorithm,'Parameters':params,'Clusters':len(set(labels)),'Noise points':0,'Silhouette':silhouette_score(X,labels)})
for eps in [0.8,1.0,1.2,1.5,1.8,2.0]:
    labels=DBSCAN(eps=eps,min_samples=5).fit_predict(X); valid=labels!=-1; ncl=len(set(labels)-{-1}); score=np.nan
    if ncl>=2 and valid.sum()>ncl: score=silhouette_score(X[valid],labels[valid])
    rows.append({'Algorithm':'DBSCAN','Parameters':f'eps={eps}, min_samples=5','Clusters':ncl,'Noise points':int((labels==-1).sum()),'Silhouette':score})
comparison=pd.DataFrame(rows); display(comparison)
# Select the highest silhouette K-Means solution among 3–6 clusters to support 3+ segment models.
best=comparison[(comparison.Algorithm=='K-Means') & (comparison.Clusters>=3)].sort_values('Silhouette',ascending=False).iloc[0]
k=int(best.Parameters.split('=')[1]); df['Segment']=KMeans(n_clusters=k,random_state=42,n_init=20).fit_predict(X)
display(df['Segment'].value_counts().sort_index())

## 3. Segment profiling

In [ ]:
profile=df.groupby('Segment').agg(Customers=('CustomerID','count'),AvgTenure=('Tenure','mean'),AvgMonthlyCharges=('MonthlyCharges','mean'),AvgTotalCharges=('TotalCharges','mean'),ChurnRate=('Churn','mean')).round(2)
profile['ChurnRatePercent']=profile['ChurnRate']*100
display(profile)

## 4. Separate Random Forest models and hyperparameter tuning

In [ ]:
results=[]
for seg in sorted(df.Segment.unique()):
    part=df[df.Segment==seg]; Xs=part[num_features+cat_features]; ys=part.Churn.astype(int)
    strat=ys if ys.nunique()==2 and ys.value_counts().min()>=2 else None
    Xtr,Xte,ytr,yte=train_test_split(Xs,ys,test_size=.25,random_state=42,stratify=strat)
    prep_model=ColumnTransformer([('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num_features),('cat',OneHotEncoder(handle_unknown='ignore'),cat_features)])
    pipe=Pipeline([('prep',prep_model),('rf',RandomForestClassifier(random_state=42,class_weight='balanced'))])
    min_class=int(ytr.value_counts().min()) if ytr.nunique()==2 else 0
    if ytr.nunique()==2 and min_class>=2 and len(ytr)>=8:
        cv=StratifiedKFold(n_splits=min(3,min_class),shuffle=True,random_state=42)
        search=GridSearchCV(pipe,{'rf__n_estimators':[100,200],'rf__max_depth':[None,5,10],'rf__min_samples_split':[2,5],'rf__min_samples_leaf':[1,2]},scoring='f1',cv=cv,n_jobs=-1)
        search.fit(Xtr,ytr); model=search.best_estimator_; cv_f1=search.best_score_; tuning='GridSearchCV'
    else:
        model=pipe.fit(Xtr,ytr); cv_f1=np.nan; tuning='Insufficient minority examples; default RF'
    pred=model.predict(Xte); prob=model.predict_proba(Xte)[:,1] if len(model.classes_)>1 else np.zeros(len(yte))
    results.append({'Segment':int(seg),'Customers':len(part),'Accuracy':accuracy_score(yte,pred),'Precision':precision_score(yte,pred,zero_division=0),'Recall':recall_score(yte,pred,zero_division=0),'F1Score':f1_score(yte,pred,zero_division=0),'ROCAUC':roc_auc_score(yte,prob) if yte.nunique()==2 else np.nan,'CVBestF1':cv_f1,'Tuning':tuning})
evaluation=pd.DataFrame(results); display(evaluation)

## 5. Business recommendations

Use segment size, customer tenure, monthly charges and churn rate to prioritize actions. Track retention outcomes against a holdout group before estimating financial impact.